In [ ]:
# adds range as a parameter to the sweep to identify ideal window sizes / resolutions

import os
import utils  # Import utility functions
import spacy
import glob
import pandas as pd
import networkx as nx
import nltk
from nltk.corpus import stopwords
import community as community_louvain
import random
import numpy as np
from collections import defaultdict
import csv
import json

#seeds random generators for reproducibility
random.seed(42)
np.random.seed(42)

# define stop words
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))  # Define stop_words

def unit_id_to_float(unit_id):
    book, unit = unit_id.split("_")
    return float(f"{book}.{int(unit):02d}")
    

# -----------sets base directory, locates txt files and reads them into texts
BASE_DIR = r"C:\Users\emine\try_env\2025_02"
directory = os.path.join(BASE_DIR, "texts")
pattern = "*.txt"
file_paths = glob.glob(os.path.join(directory, pattern))
texts = utils.read_text_files(directory)
export_dir = os.path.join(BASE_DIR, "hierarchical", "hierarchical_ranged", "parameter_sweep")
# -----------sets base directory, locates txt files and reads them into texts

# ----------- loading headword  & proper-nouns lists, lemmatizing them for consistency
lists_directory = os.path.join(BASE_DIR, "lists")
proper_nouns = utils.load_proper_nouns(os.path.join(lists_directory, "mod_proper_nouns_misspelled.csv"))
headwords_one = utils.load_word_list(os.path.join(lists_directory, "BNC-COCA_headwords_1000.txt"))
headwords_two = utils.load_word_list(os.path.join(lists_directory, "BNC-COCA_headwords_2000.txt"))
headwords_three = utils.load_word_list(os.path.join(lists_directory, "BNC-COCA_headwords_3000.txt"))
list1_words = utils.lemmatize_word_list(headwords_one)  # Initialize word_list with list1_words
list2_words = utils.lemmatize_word_list(headwords_two)  # Initialize word_list with list2_words
list3_words = utils.lemmatize_word_list(headwords_three)  # Initialize word_list with list3_words
# ----------- loading headword  & proper-nouns lists, lemmatizing them for consistency


#----------- Compute word frequencies
word_frequencies = utils.compute_word_frequencies(texts, proper_nouns, stop_words)
threshold = 60
high_frequency_words = [word for word, freq in word_frequencies.items() if freq > threshold]
# Update the most common words to be excluded in the graphs by updating stop words!
stop_words.update(high_frequency_words)
#----------- Compute word frequencies

print(f"Words occurring more than {threshold} times:")
print(high_frequency_words)

#----------- # Initialize data structures
snapshots = {}  
#global_cooccurrence_counts = defaultdict(lambda: defaultdict(int))
#unit_graphs = defaultdict(dict)
#token_first_appearance = {}
modularity_scores = {}


# --- Define custom time windows for each book ---
time_windows = []
unit_interval_map = {}  # Initialize unit_interval_map
unit_topic_rows = []  # For storing topic distribution per unit

book_ranges = {
    1: range(1, 19),  # Book 1: 1_1_More to 1_18_More
    2: range(1, 21),  # Book 2: 2_1_More to 2_20_More
    3: range(1, 14),  # Book 3: 3_1_More to 3_13_More
    4: range(1, 14)   # Book 4: 4_1_More to 4_13_More
}

#----------- Populate snapshots with file paths
for book, units in book_ranges.items():
    snapshots[book] = []
    for unit in units:
        file_name = f"{book}_{unit}_More.txt"
        file_path = os.path.join(directory, file_name)
        if os.path.exists(file_path):
            snapshots[book].append(file_path)
        else:
            print(f"File not found: {file_path}")
#----------- org text files by book and unit, stored in snapshots

# Define time windows for each book
for book, units in book_ranges.items():
    unit_floats = [unit_id_to_float(f"{book}_{unit}") for unit in units]
    unit_floats.sort()
    if unit_floats:
        mid_index = len(unit_floats) // 2
        part1 = unit_floats[:mid_index]
        part2 = unit_floats[mid_index:]
        if part1:
            time_windows.append((part1[0], part1[-1]))
        if part2:
            time_windows.append((part2[0], part2[-1]))


# Create lookup from unit_id -> (start, end)
for start, end in time_windows:
    for book, units in book_ranges.items():
        for unit in units:
            uid = f"{book}_{unit}"
            uid_float = unit_id_to_float(uid)
            if start <= uid_float <= end:
                unit_interval_map[uid] = (start, end)
                
# Now `unit_interval_map` is defined and populated
import os
import networkx as nx
from collections import defaultdict
from community import community_louvain

# … assume utils, book_ranges, unit_id_to_float, proper_nouns, stop_words,
#     directory, snapshots init code, etc. are already up here …

def build_global_graph(window_size):
    """
    Build unit_graphs, global_cooccurrence_counts, and the global_graph
    for a given window_size. Returns (global_graph, token_first_appearance, unit_graphs).
    """
    token_first_appearance = {}
    global_cooccurrence_counts = defaultdict(lambda: defaultdict(int))
    unit_graphs = {}
    word_unit_occurrences = defaultdict(set)  # Track word → set of unit_ids

    for book, file_paths in snapshots.items():
        for file_path in file_paths:
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
            words = utils.filter_text(text, proper_nouns, stop_words)
            if not words:
                continue

            adj_matrix, unique_words = utils.build_cooccurrence_matrix(words, window_size)
            filename = os.path.basename(file_path)
            uid = "_".join(filename.split("_")[:2])  # e.g. "1_4"

            # track first appearance
            for w in unique_words:
                token_first_appearance.setdefault(w, uid)
                word_unit_occurrences[w].add(uid)  # <-- Record where word appears

            # build unit graph (optional, used downstream)
            unit_graph = nx.Graph()
            unit_graph.add_nodes_from(unique_words)
            for i, wi in enumerate(unique_words):
                for j, wj in enumerate(unique_words):
                    wt = adj_matrix[i, j]
                    if wt:
                        unit_graph.add_edge(wi, wj, weight=wt)
                    if wi != wj and wt:
                        global_cooccurrence_counts[wi][wj] += wt
            unit_graphs[uid] = unit_graph

    # Build global graph
    G = nx.Graph()
    for w in sorted(global_cooccurrence_counts):
        G.add_node(w, source=token_first_appearance.get(w, "unknown"))

    for wi, nbrs in global_cooccurrence_counts.items():
        for wj, wt in nbrs.items():
            if wt > 0:
                G.add_edge(wi, wj, weight=wt)

    # Add 'start' and 'range' attributes
    for word, data in G.nodes(data=True):
        uid = data.get("source", "unknown")
        if isinstance(uid, str) and "_" in uid:
            G.nodes[word]['start'] = unit_id_to_float(uid)
        G.nodes[word]['range'] = len(word_unit_occurrences.get(word, []))  # Add range

    # Rescale edge weights based on node-level range (to demote extreme cases)
    MIN_RANGE = 3
    MAX_RANGE = len(unit_graphs) * 0.9  # Assumes you're calling this before sweep sets total_units
    EPSILON = 1e-6
    
    def range_scaling(w):
        r = G.nodes[w].get("range", 0)
        if r < MIN_RANGE or r > MAX_RANGE:
            return 0.25  # Downweight nodes that are too narrow or too broad
        return 1.0
    
    # Rescale edge weights in-place
    for u, v, data in G.edges(data=True):
        original_weight = data.get("weight", 1.0)
        scale_u = range_scaling(u)
        scale_v = range_scaling(v)
        scaling = (scale_u + scale_v) / 2
        G[u][v]['weight'] = original_weight * scaling


    return G, token_first_appearance, unit_graphs

# ─────────────────────────────────────────────────────────────────────────────
# 4) Parameter sweep: window_size × resolution
best_combo = {
    "window_size": None,
    "resolution":  None,
    "modularity":  -1.0,
    "avg_range":   -1.0,
    "score":       -1.0,
    "graph":       None,
    "partition":   None
}

# Normalize this according to your actual corpus size
total_units = len(unit_graphs) if 'unit_graphs' in locals() else 1
max_possible_range = total_units  # Normalize range to [0, 1]


for ws in [3, 4, 5, 6, 7, 8, 9, 10]:
    print(f"\n=== Testing window_size = {ws} ===")
    G_ws, token_first_appearance, unit_graphs_ws = build_global_graph(ws)

    for res in [0.2, 0.4, 0.6, 0.8, 1.0]:
        part = community_louvain.best_partition(G_ws, resolution=res)
        mod  = community_louvain.modularity(part, G_ws)

        # ---- Compute topic-level average range
        topic_to_words = defaultdict(list)
        for word, topic_id in part.items():
            topic_to_words[topic_id].append(word)

        topic_avg_ranges = []
        for words in topic_to_words.values():
            ranges = [G_ws.nodes[w].get("range", 0) for w in words]
            if ranges:
                topic_avg_ranges.append(np.mean(ranges))

        avg_range = np.mean(topic_avg_ranges) if topic_avg_ranges else 0
        norm_range = avg_range / max_possible_range

        # ---- Combine modularity + range into a single score
        alpha, beta = 0.7, 0.3  # weight modularity vs. range
        score = alpha * mod + beta * norm_range

        print(f"  res={res:>3} → modularity={mod:.4f}, avg_range={avg_range:.2f}, score={score:.4f}")

        if score > best_combo["score"]:
            best_combo.update(
                window_size=ws,
                resolution=res,
                modularity=mod,
                avg_range=avg_range,
                score=score,
                graph=G_ws,
                partition=part
            )

# 5) Report best combo
print(f"\n🏆 Best combo:")
print(f"   window_size = {best_combo['window_size']}")
print(f"   resolution  = {best_combo['resolution']}")
print(f"   modularity  = {best_combo['modularity']:.4f}")
print(f"   avg_range   = {best_combo['avg_range']:.2f}")
print(f"   combined score = {best_combo['score']:.4f}")


# Extract winning graph & partition
global_graph     = best_combo["graph"]
global_partition = best_combo["partition"]
word_to_topic    = {w: t for w, t in global_partition.items()}

# ─────────────────────────────────────────────────────────────────────────────
# 6) Annotate & export using your existing palette + helper

hex_palette = [
    "#e6194b", "#3cb44b", "#722dd0", "#4363d8",
    "#f58231", "#cc3300", "#46f0f0", "#0fa9d0",
    "#384a03", "#663300", "#008080", "#6e00b3",
    "#9a6324", "#0fa929", "#800000", "#3333ff",
]
topic_colors = {tid: hex_palette[tid] for tid in range(len(hex_palette))}

def annotate_with_colors(G):
    for n, d in G.nodes(data=True):
        t = d.get('topic')
        hexcol = topic_colors.get(t, "#CCCCCC")
        r, g, b = (int(hexcol.lstrip('#')[i:i+2], 16) for i in (0,2,4))
        G.nodes[n].update(color=hexcol, viz={'color': {'r':r,'g':g,'b':b,'a':1.0}})


# assign & color global
for w, t in global_partition.items():
    global_graph.nodes[w]['topic'] = t

# ─── assign shapes by list‐membership ───────────────────────────────────
for w in global_graph.nodes:
    if w in list1:
        shp = "triangle"
    elif w in list2:
        shp = "square"
    elif w in list3:
        shp = "star"
    else:
        shp = "circle"
    global_graph.nodes[w]['shape'] = shp
# ────────────────────────────────────────────────────────────────────────

# now color‐annotate as before
annotate_with_colors(global_graph)

# write global
out_global = os.path.join(export_dir, "global_best.gexf")
nx.write_gexf(global_graph, out_global)
print(f"Saved winning global graph → {out_global}")

# 7) write one subgraph per topic
for topic_id in sorted(set(word_to_topic.values())):
    nodes = [n for n,d in global_graph.nodes(data=True) if d['topic']==topic_id]
    Gt = global_graph.subgraph(nodes).copy()
    out_t = os.path.join(export_dir, f"topic_{topic_id}_subgraph.gexf")
    nx.write_gexf(Gt, out_t)
    print(f"Saved topic {topic_id} subgraph → {out_t}")


In [ ]:
# adds range as a parameter to the sweep to identify ideal window sizes / resolutions

import os
import utils  # Import utility functions
import spacy
import glob
import pandas as pd
import networkx as nx
import nltk
from nltk.corpus import stopwords
import community as community_louvain
import random
import numpy as np
from collections import defaultdict
import csv
import json

#seeds random generators for reproducibility
random.seed(42)
np.random.seed(42)

# define stop words
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))  # Define stop_words

def unit_id_to_float(unit_id):
    book, unit = unit_id.split("_")
    return float(f"{book}.{int(unit):02d}")
    

# -----------sets base directory, locates txt files and reads them into texts
BASE_DIR = r"C:\Users\emine\try_env\2025_02"
directory = os.path.join(BASE_DIR, "texts")
pattern = "*.txt"
file_paths = glob.glob(os.path.join(directory, pattern))
texts = utils.read_text_files(directory)
export_dir = os.path.join(BASE_DIR, "hierarchical", "hierarchical_ranged", "parameter_sweep")
# -----------sets base directory, locates txt files and reads them into texts

# ----------- loading headword  & proper-nouns lists, lemmatizing them for consistency
lists_directory = os.path.join(BASE_DIR, "lists")
proper_nouns = utils.load_proper_nouns(os.path.join(lists_directory, "mod_proper_nouns_misspelled.csv"))
headwords_one = utils.load_word_list(os.path.join(lists_directory, "BNC-COCA_headwords_1000.txt"))
headwords_two = utils.load_word_list(os.path.join(lists_directory, "BNC-COCA_headwords_2000.txt"))
headwords_three = utils.load_word_list(os.path.join(lists_directory, "BNC-COCA_headwords_3000.txt"))
list1_words = utils.lemmatize_word_list(headwords_one)  # Initialize word_list with list1_words
list2_words = utils.lemmatize_word_list(headwords_two)  # Initialize word_list with list2_words
list3_words = utils.lemmatize_word_list(headwords_three)  # Initialize word_list with list3_words
# ----------- loading headword  & proper-nouns lists, lemmatizing them for consistency


#----------- Compute word frequencies: counts words across the 
#entire corpus, excl. proper nouns and stopwords; finds words appearing 60+ times
# adds them to stop_words to exclude from the network
#prevent ultra-frequent items from dominating co-occurrence structure
word_frequencies = utils.compute_word_frequencies(texts, proper_nouns, stop_words)
threshold = 60
high_frequency_words = [word for word, freq in word_frequencies.items() if freq > threshold]
# Update the most common words to be excluded in the graphs by updating stop words!
stop_words.update(high_frequency_words)
#----------- Compute word frequencies

print(f"Words occurring more than {threshold} times:")
print(high_frequency_words)

#----------- # Initialize data structures
snapshots = {}  
#global_cooccurrence_counts = defaultdict(lambda: defaultdict(int))
#unit_graphs = defaultdict(dict)
#token_first_appearance = {}
modularity_scores = {}


# --- Define custom time windows for each book ---
time_windows = []
unit_interval_map = {}  # Initialize unit_interval_map
#unit_topic_rows = []  # For storing topic distribution per unit

#defines the units that exist for each book
book_ranges = {
    1: range(1, 19),  # Book 1: 1_1_More to 1_18_More
    2: range(1, 21),  # Book 2: 2_1_More to 2_20_More
    3: range(1, 14),  # Book 3: 3_1_More to 3_13_More
    4: range(1, 14)   # Book 4: 4_1_More to 4_13_More
}

#----------- Populate snapshots with file paths: 
#for every expected unit ifle, construct a file name
#if it exists, store it under that book in snapshots
#otherwise print a warning
for book, units in book_ranges.items():
    snapshots[book] = []
    for unit in units:
        file_name = f"{book}_{unit}_More.txt"
        file_path = os.path.join(directory, file_name)
        if os.path.exists(file_path):
            snapshots[book].append(file_path)
        else:
            print(f"File not found: {file_path}")

# Define the maximum possible range based on the units we actually loaded
max_possible_range = sum(len(paths) for paths in snapshots.values())

#----------- org text files by book and unit, stored in snapshots

# Define time windows for each book
# create time windows by splitting each book in half
# converts each unit into a float (1.01, 1.02)
# splits it into two halves
# adds two windows per book: early half and late half
for book, units in book_ranges.items():
    unit_floats = [unit_id_to_float(f"{book}_{unit}") for unit in units]
    unit_floats.sort()
    if unit_floats:
        mid_index = len(unit_floats) // 2
        part1 = unit_floats[:mid_index]
        part2 = unit_floats[mid_index:]
        if part1:
            time_windows.append((part1[0], part1[-1]))
        if part2:
            time_windows.append((part2[0], part2[-1]))


# Create lookup from unit_id -> (start, end)
# for each defined window, check with unit floats fall inside it
# and store the mapping in unit_interval_map
# currently not in use!!!-
for start, end in time_windows:
    for book, units in book_ranges.items():
        for unit in units:
            uid = f"{book}_{unit}"
            uid_float = unit_id_to_float(uid)
            if start <= uid_float <= end:
                unit_interval_map[uid] = (start, end)
                
# Now `unit_interval_map` is defined and populated

# --- build co-occurrence graphs
#defines a function that builds unit-level graphs (per file) and a global graph merging them

def build_global_graph(window_size):
    """
    Build unit_graphs, global_cooccurrence_counts, and the global_graph
    for a given window_size. Returns (global_graph, token_first_appearance, unit_graphs).
    """
    token_first_appearance = {}
    global_cooccurrence_counts = defaultdict(lambda: defaultdict(int))
    unit_graphs = {}
    word_unit_occurrences = defaultdict(set)  # Track word → set of unit_ids
    #loop over all units by reading the text file, filtering it (depending on utils!) and skipping empty results
    # which units end up being empty??
    for book, file_paths in snapshots.items():
        for file_path in file_paths:
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
            words = utils.filter_text(text, proper_nouns, stop_words)
            if not words:
                continue

            #collect unique tokens, count co-occurrences with window-size tokens,
            # and return adjacency matrix + word list
            adj_matrix, unique_words = utils.build_cooccurrence_matrix(words, window_size)
            filename = os.path.basename(file_path)
            uid = "_".join(filename.split("_")[:2])  # e.g. "1_4"

            # track first appearance and range by adding the unit ID to the set to know it appeared here
            for w in unique_words:
                token_first_appearance.setdefault(w, uid)
                word_unit_occurrences[w].add(uid)  # <-- Record where word appears

            # build unit graph and update global counts
            # creates a networkX graph for the unit
            # adds all unique words as nodes
            # for every pair, if adjacency weight exists: 1) add edge in unit graph
            # and 2) add to the global co-occurrence counts
        
            unit_graph = nx.Graph()
            unit_graph.add_nodes_from(unique_words)
            for i, wi in enumerate(unique_words):
                for j, wj in enumerate(unique_words):
                    wt = adj_matrix[i, j]
                    if wt:
                        unit_graph.add_edge(wi, wj, weight=wt)
                    if wi != wj and wt:
                        global_cooccurrence_counts[wi][wj] += wt
            unit_graphs[uid] = unit_graph

    # Build global graph
    # created global graph and adds nodes
    # each node has attribute source to account for its first unit of appearance
    G = nx.Graph()
    for w in sorted(global_cooccurrence_counts):
        G.add_node(w, source=token_first_appearance.get(w, "unknown"))

    for wi, nbrs in global_cooccurrence_counts.items():
        for wj, wt in nbrs.items():
            if wt > 0:
                G.add_edge(wi, wj, weight=wt)

    # Add 'start' and 'range' attributes
                # start: float value for the 1st appearance-unit
                #range: nr of unique units the word appears in
    for word, data in G.nodes(data=True):
        uid = data.get("source", "unknown")
        if isinstance(uid, str) and "_" in uid:
            G.nodes[word]['start'] = unit_id_to_float(uid)
        G.nodes[word]['range'] = len(word_unit_occurrences.get(word, []))  # Add range

    # Rescale edge weights based on node-level range (to demote extreme cases)
    MIN_RANGE = 3
    MAX_RANGE = len(unit_graphs) * 0.9  # Assumes you're calling this before sweep sets total_units
    EPSILON = 1e-6

    #if a word appears in too few or too many units, scale its edges down.
    def range_scaling(w):
        r = G.nodes[w].get("range", 0)
        if r < MIN_RANGE or r > MAX_RANGE:
            return 0.25  # Downweight nodes that are too narrow or too broad
        return 1.0
    
    # Rescale edge weights in-place by average scaling of its endpoints
    for u, v, data in G.edges(data=True):
        original_weight = data.get("weight", 1.0)
        scale_u = range_scaling(u)
        scale_v = range_scaling(v)
        scaling = (scale_u + scale_v) / 2
        G[u][v]['weight'] = original_weight * scaling

    # returns global graph + metadata
    return G, token_first_appearance, unit_graphs
    

# ─────────────────────────────────────────────────────────────────────────────

# Fix parameters 
WINDOW_SIZE = 5
RESOLUTION = 1.0
#Build grpah for the window size
G, token_first_appearance, unit_graphs = build_global_graph(WINDOW_SIZE)

# calculates range value
total_units = sum(len(paths) for paths in snapshots.values())
max_possible_range = total_units

# Rund Louvain  
print(f"Running Louvain with resolution = {RESOLUTION}")
partition = community_louvain.best_partition(
    G,
    weight='weight',
    resolution=RESOLUTION,
    randomize=False
)
mod = community_louvain.modularity(partition, G)


# ---- Compute topic-level average range
topic_to_words = defaultdict(list)
for word, topic_id in partition.items():
    topic_to_words[topic_id].append(word)

topic_avg_ranges = []
for words in topic_to_words.values():
    # ensure the block uses same graph used to compute the partition
    ranges = [G.nodes[w]["range"] for w in words if w in G]
    if ranges:
        topic_avg_ranges.append(np.mean(ranges))
# to avoid skewing of results due to overly small topics generated
# fix skewing by weighitng by topic size (i.e. larger topics count more)
topic_avg_ranges = []
weights = []
for words in topic_to_words.values():
    ranges = [G.nodes[w]["range"] for w in words]
    topic_avg_ranges.append(np.mean(ranges))
    weights.append(len(words))
avg_range = np.average(topic_avg_ranges, weights=weights)



norm_range = avg_range / max_possible_range

# ---- Combine modularity + range into a single score
alpha, beta = 0.7, 0.3  # weight modularity vs. range
score = alpha * mod + beta * norm_range
# tries to group words by community/topic, computer average "range" within each topic
# combine modularity and range into a single score
print(f"  res={RESOLUTION} → modularity={mod:.4f}, avg_range={avg_range:.2f}, score={score:.4f}")


# Extract winning graph & partition, set names for later steps of exports and coloring
global_graph     = G
global_partition = partition
word_to_topic    = {w: t for w, t in partition.items()}

# ─────────────────────────────────────────────────────────────────────────────
# 6) Annotate & export using your existing palette + helper

hex_palette = [
    "#e6194b", "#3cb44b", "#722dd0", "#4363d8",
    "#f58231", "#cc3300", "#46f0f0", "#0fa9d0",
    "#384a03", "#663300", "#008080", "#6e00b3",
    "#9a6324", "#0fa929", "#800000", "#3333ff",
]
topic_colors = {tid: hex_palette[tid] for tid in range(len(hex_palette))}

# coloring nodes: for each node, read its topic, choose a hex color
# convert hex color to RGB integers
# store color info in color and viz fields
def annotate_with_colors(G):
    for n, d in G.nodes(data=True):
        t = d.get('topic')
        hexcol = topic_colors.get(t, "#CCCCCC")
        r, g, b = (int(hexcol.lstrip('#')[i:i+2], 16) for i in (0,2,4))
        G.nodes[n].update(color=hexcol, viz={'color': {'r':r,'g':g,'b':b,'a':1.0}})


# assign & color global: assign topics to nodes, writes each node's community id into its node attributes
for w, t in partition.items():
    global_graph.nodes[w]['topic'] = t

# ─── assign shapes by list‐membership ───────────────────────────────────
for w in global_graph.nodes:
    if w in list1_words:
        shp = "triangle"
    elif w in list2_words:
        shp = "square"
    elif w in list3_words:
        shp = "star"
    else:
        shp = "circle"
    global_graph.nodes[w]['shape'] = shp
# ────────────────────────────────────────────────────────────────────────

# now applies colors and exports to global graph
annotate_with_colors(global_graph)

# write global
out_global = os.path.join(export_dir, "global_best.gexf")
nx.write_gexf(global_graph, out_global)
# and writes the global graph to a .gexf file for Gephi
print(f"Saved winning global graph → {out_global}")

# 7) write one subgraph per topic: for each community
# select nodes in that community
# create a copy of that subgraph
# export it as a separate GEXF
for topic_id in sorted(set(word_to_topic.values())):
    nodes = [n for n,d in global_graph.nodes(data=True) if d['topic']==topic_id]
    Gt = global_graph.subgraph(nodes).copy()
    out_t = os.path.join(export_dir, f"topic_{topic_id}_subgraph.gexf")
    nx.write_gexf(Gt, out_t)
    print(f"Saved topic {topic_id} subgraph → {out_t}")

import pickle
# load reference texts and build reference lemma set
REF_DIR = os.path.join(BASE_DIR, "reference_texts")
ref_texts = utils.read_text_files(REF_DIR)

# Build ref_lemmas i.e. a reference corpus
ref_lemmas = set()
for text in ref_texts:
    tokens = utils.filter_text(text, proper_nouns, stop_words)
    ref_lemmas.update(tokens)
# filters in the smae way and stores in a set

# Serialize to disk: Saves the graph and supporting objects to disk in a single file (state.pkl) so you can reload without recomputing.
state_path = os.path.join(export_dir, "state.pkl")
with open(state_path, "wb") as f:
    pickle.dump({
        "global_graph": global_graph,
        "snapshots":    snapshots,
        "ref_lemmas":   ref_lemmas
    }, f)

print(f"Serialized pipeline state → {state_path}")